In [1]:
import numpy as np
import adi
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import find_peaks

sample_rate = 6e6 # Hz
center_freq = 1420.405751766e6 # Hz
num_samps = 16384# number of samples per call to rx()

sdr = adi.Pluto("ip:192.168.2.1")
sdr.sample_rate = int(sample_rate)

# Config Rx (setting for Rxpin)
sdr.rx_lo = int(center_freq) # set what center frequency you want
sdr.rx_rf_bandwidth = int(sample_rate) #how much range you want
sdr.rx_buffer_size = num_samps # how many samples you want
sdr.gain_control_mode_chan0 = 'manual'
sdr.rx_hardwaregain_chan0 = 50.0 # dB, increase to increase the receive gain, but be careful not to saturate the ADC

# Receive samples
rx_samples = sdr.rx() # to receive radio
print(rx_samples) # to show what raw data you collected

# Calculate power spectral density (frequency domain version of signal)
psd = np.abs(np.fft.fftshift(np.fft.fft(rx_samples)))**2 # this codes are to perform a FFT to convert time into frequency data, rearrange the data horizontally so that the center of the graph is at 0Hz, and square the complex-valued data to calculate the power of radio wave (PSD).
psd_dB = 10*np.log10(psd) #transform the strength of radio to dB
f = np.linspace(sample_rate/-2, sample_rate/2, len(psd)) # create a list of numbers to serve as the horizontal axis (frequency axis) of the graph


# Plot time domain
#plt.figure(0) #Prepare the window for the new graph (index 0).
#plt.plot(np.real(rx_samples[::100])) #It plots the **real part (I-signal)** of the received data. The `::100` command displays every 100th point to reduce the data size and improve performance.
#plt.plot(np.imag(rx_samples[::100])) # for imaginary number
#plt.xlabel("Time")
#plt.title("Plot for time domain")

# Plot freq domain
plt.figure(1)
plt.title("Plot for frequency domain")
plt.plot(f/1e6, psd_dB)
plt.xlabel("Frequency [MHz]")
plt.ylabel("PSD") 
plt.show()

#Plot Spike freq
peaks, _ = find_peaks(psd_dB, height=85, distance = 500)

observed_freq = center_freq + f[peaks]

z=(center_freq-observed_freq)/observed_freq #equation for redshift/blueshift

c=299792458 #speed of light in m

wavelength=c/observed_freq

spike_data = { 'Spike_Frequency(Hz)' : observed_freq,
               'Spike_PSD(dB)' : psd_dB[peaks],
               'status' :['redshift' if val>0 else 'blueshift' if val<0 else 'No shift' for val in z],
               'wavelength(m)' : wavelength,
               'velocity_of_star(km/s)' : z*c/1000
             }

df_spikes = pd.DataFrame(spike_data)

if df_spikes.empty:
    print("No radio spike detected")

else: 
    plt.figure(1)
    plt.title("Detected spikes from frequency domain's plot")
    plt.plot(f[peaks], psd_dB[peaks], "o", label="Detected Spikes")
    plt.xlabel("Frequency [MHz]")
    plt.ylabel("PSD") 
    plt.show()
    print(df_spikes)
    print()
    print(f"z = {z}")
    print()
    print("When z is positive value, redshift happens. And when z is negative value, blueshift happens.")

print()
frequency_resolution=(sample_rate)/(num_samps)

print(f"The frequency resolution is {frequency_resolution} Hz")


Exception: No device found